In [2]:
import scanpy as sc
import pandas as pd  
import numpy as np
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts


In [3]:
atlas_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/atlas_data.h5ad")

In [4]:
query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/harmonized_nucleus_data.h5ad")

### set labels to use for integration:

check which cells have only level 1 annotations:

In [5]:
lev1_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 1, :].copy()
print("Number of cells with only level 1 annotation:", lev1_obs.shape[0])

Number of cells with only level 1 annotation: 2815


In [6]:
# count number of cells per ann_level_1 group that have no further annotations
lev1_annotations = lev1_obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).rename(columns={"ann_level_1":"n_cells_w_only_l1_ann"})
# count total cells in each ann_level_1 group
lev1_annotations['total'] = query_data.obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).loc[lev1_annotations.index,"ann_level_1"]

In [7]:
lev1_annotations

,n_cells_w_only_l1_ann,total
ann_level_1,,
Endothelial,0,21239
Epithelial,0,59585
Immune,132,37881
Stroma,0,18794
Unknown,2683,2683


Not applicable here. (We will let all "Proliferating cells" keep their labels, and will set all other cells as "unlabeled" (since they only make up a small part of their group))

In [8]:
lev1_labeled = []

In [9]:
lev1_unlabeled = query_data.obs.index[[max_res == 1 for max_res, lev1label in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_1)]]

In [10]:
len(lev1_labeled)

0

In [11]:
len(lev1_unlabeled)

2815

In [12]:
# check if all labels/cells are accounted for
lev1_annotations.n_cells_w_only_l1_ann.sum() == len(lev1_labeled) + len(lev1_unlabeled)

True

Now check which cells have only up to level 2 annotations:

In [13]:
lev2_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 2, :].copy()
print("Number of cells with only level 1/2 annotation:", lev2_obs.shape[0])

Number of cells with only level 1/2 annotation: 8010


In [14]:
# count number of cells per ann_level_2 group that have no further annotations
lev2_annotations = lev2_obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).rename(columns={"ann_level_2":"n_cells_w_only_l2_ann"})
# count total cells in each ann_level_2 group
lev2_annotations['total'] = query_data.obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).loc[lev2_annotations.index,"ann_level_2"]

In [15]:
lev2_annotations

,n_cells_w_only_l2_ann,total
ann_level_2,,
1_Immune,0,132
1_Unknown,0,2683
Airway epithelium,0,9155
Alveolar epithelium,0,50382
Blood vessels,4159,19336
Chondrocytes,394,394
Fibroblast lineage,1554,17405
Granulocytes,0,2436
Lymphatic EC,1903,1903


We will set all cells with level 2 annotations "Chondrocytes", "Fibroblast lineage", "Lymphatic EC", "Smooth muscle" to their level 2 annotations.

In [16]:
lev2_labels_to_keep = ["Chondrocytes", "Fibroblast lineage", "Lymphatic EC", "Smooth muscle"]
lev2_labeled = query_data.obs.index[[lev2_lab in lev2_labels_to_keep for lev2_lab in query_data.obs.ann_level_2]]

In [17]:
len(lev2_labeled) == lev2_annotations.loc[lev2_labels_to_keep,"total"].sum()

True

In [18]:
lev2_unlabeled = query_data.obs.index[[max_res == 2 and lev2_lab not in lev2_labels_to_keep for max_res, lev2_lab in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_2)]]

In [19]:
len(lev2_unlabeled) == lev2_annotations.loc[~lev2_annotations.index.isin(lev2_labels_to_keep),"n_cells_w_only_l2_ann"].sum()

True

now generate scANVI labels:

In [20]:
query_data.obs['scanvi_label'] = query_data.obs.ann_level_3.tolist()
query_data.obs.loc[lev1_labeled,'scanvi_label'] = query_data.obs.loc[lev1_labeled, "ann_level_1"]
query_data.obs.loc[lev1_unlabeled,'scanvi_label'] = "unlabeled"
query_data.obs.loc[lev2_labeled,'scanvi_label'] = query_data.obs.loc[lev2_labeled, "ann_level_2"]
query_data.obs.loc[lev2_unlabeled,'scanvi_label'] = "unlabeled"

In [21]:
# Check which label in query data is new and not present in reference atlas
[x for x in query_data.obs['scanvi_label'].unique() if x not in atlas_data.obs["scanvi_label"].unique()]

['BASCs', 'Erythrocytes', 'Chondrocytes']

In [22]:
query_data.obs['scanvi_label'] = query_data.obs['scanvi_label'].astype("category")

save dataset

In [23]:
query_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/nucleus_labeled.h5ad")

... storing 'ann_level_5_clean' as categorical


In [24]:
#query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/nucleus_labeled.h5ad")

### Normalize counts

In [25]:
#Save raw counts
query_data.layers["counts"] = query_data.X.copy()

In [26]:
query_data.obs.dataset.unique()

['Eils_2020', 'Duong_lungMAP_unpubl', 'Sun_2020_batch1', 'Sun_2020_batch2', 'Sun_2020_batch3', 'Sun_2020_batch4']
Categories (6, object): ['Eils_2020', 'Duong_lungMAP_unpubl', 'Sun_2020_batch1', 'Sun_2020_batch2', 'Sun_2020_batch3', 'Sun_2020_batch4']

Normalize each dataset on its own to save time. As normalization is used in "batches" this is ok to do.# Check which label in query data is new and not present in reference atlas

In [27]:
lukassen = query_data[(query_data.obs.dataset=="Eils_2020"),:]
duong = query_data[(query_data.obs.dataset=="Duong_lungMAP_unpubl"),:]
wang_sub_1 = query_data[(query_data.obs.dataset=="Sun_2020_batch1"),:]
wang_sub_2 = query_data[(query_data.obs.dataset=="Sun_2020_batch2"),:]
wang_sub_3 = query_data[(query_data.obs.dataset=="Sun_2020_batch3"),:]
wang_sub_4 = query_data[(query_data.obs.dataset=="Sun_2020_batch4"),:]

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)


In [28]:
# normalize 
lukassen_norm = scib_excerpts.SCRAN_normalize(lukassen, log_transform=False)

R[write to console]: Loading required package: SingleCellExperiment

R[write to console]: Loading required package: SummarizedExperiment

R[write to console]: Loading required package: MatrixGenerics

R[write to console]: Loading required package: matrixStats

R[write to console]: 
Attaching package: ‘MatrixGenerics’


R[write to console]: The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowC

In [29]:
# store
lukassen_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/lukassen_norm.h5ad")

In [30]:
# normalize 
duong_norm = scib_excerpts.SCRAN_normalize(duong, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [31]:
# store
duong_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/duong_norm.h5ad")

In [32]:
# normalize 
wang_sub_1_norm = scib_excerpts.SCRAN_normalize(wang_sub_1, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [33]:
# store
wang_sub_1_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/wang_sub_1_norm.h5ad")

In [34]:
# normalize 
wang_sub_2_norm = scib_excerpts.SCRAN_normalize(wang_sub_2, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [35]:
# store
wang_sub_2_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/wang_sub_2_norm.h5ad")

In [36]:
# normalize 
wang_sub_3_norm = scib_excerpts.SCRAN_normalize(wang_sub_3, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [37]:
# store
wang_sub_3_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/wang_sub_3_norm.h5ad")

In [38]:
# normalize 
wang_sub_4_norm = scib_excerpts.SCRAN_normalize(wang_sub_4, log_transform=False)

/opt/python/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
R[write to console]: dimnames(.) <- NULL:  translated to 
dimnames(.) <- list(NULL,NULL)  <==>  unname(.)



In [39]:
# store
wang_sub_4_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/wang_sub_4_norm.h5ad")

In [40]:
final_tmp1 = lukassen_norm.concatenate(duong_norm)
final_tmp2 = final_tmp1.concatenate(wang_sub_1_norm)
final_tmp3 = final_tmp2.concatenate(wang_sub_2_norm)
final_tmp4 = final_tmp3.concatenate(wang_sub_3_norm)
final = final_tmp4.concatenate(wang_sub_4_norm)

In [41]:
# store
final.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/nucleus_labeled_normalized.h5ad")

... storing 'dataset' as categorical
... storing 'study' as categorical
... storing 'original_celltype_ann' as categorical
... storing 'subject_ID' as categorical
... storing 'sample' as categorical
... storing 'single_cell_platform' as categorical
... storing 'sample_type' as categorical
... storing 'age' as categorical
... storing 'sex' as categorical
... storing 'ethnicity' as categorical
... storing 'smoking_status' as categorical
... storing 'anatomical_region_level_1' as categorical
... storing 'anatomical_region_coarse' as categorical
... storing 'anatomical_region_detailed' as categorical
... storing 'genome' as categorical
... storing 'ann_level_1' as categorical
... storing 'ann_level_2' as categorical
... storing 'ann_level_3' as categorical
... storing 'ann_level_4' as categorical
... storing 'ann_level_5' as categorical
... storing 'ann_level_1_clean' as categorical
... storing 'ann_level_2_clean' as categorical
... storing 'ann_level_3_clean' as categorical
... storing 'a

In [2]:
final = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/nucleus_labeled_normalized.h5ad")